# NeoScreen — MobileNetV3 Training on Google Colab (Free T4 GPU)

Run cells top-to-bottom. Expected total time: **2–3 hours** on a free T4.

**Output:** `neoscreen_v1.tflite` (~4 MB) — download and add to `flutter/assets/models/`

In [ ]:
# ── 1. Mount Drive & clone repo ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/YOUR_USERNAME/neoscreen.git
%cd neoscreen
!pip install -q albumentations tensorflow

In [ ]:
# ── 2. Dataset structure expected ─────────────────────────────────────────────
# dataset/
#   Low/      ← sclera images with low bilirubin
#   Medium/   ← borderline cases
#   High/     ← high bilirubin / jaundice
#
# Option A: copy your dataset from Drive
# !cp -r /content/drive/MyDrive/neoscreen_dataset ./dataset
#
# Option B: download Bilicam sample (public, UW)
# !wget -q <bilicam_url> -O bilicam.zip && unzip -q bilicam.zip -d dataset
#
# Option C: quick smoke-test with synthetic data (no real images needed)
import os, numpy as np
from PIL import Image

for cls, rgb in [('Low', (255,255,220)), ('Medium', (255,230,140)), ('High', (255,200,60))]:
    os.makedirs(f'dataset/{cls}', exist_ok=True)
    for i in range(60):
        noise = np.random.randint(-20, 20, (224, 224, 3), dtype=np.int16)
        arr = np.clip(np.array(rgb, dtype=np.int16) + noise, 0, 255).astype(np.uint8)
        Image.fromarray(np.broadcast_to(arr, (224, 224, 3)).copy()).save(f'dataset/{cls}/{i:04d}.jpg')

print('Dataset ready:')
for c in ['Low','Medium','High']:
    print(f'  {c}: {len(os.listdir(f"dataset/{c}"))} images')

In [ ]:
# ── 3. Train ──────────────────────────────────────────────────────────────────
!python ml/train.py --data_dir ./dataset --epochs 25 --output neoscreen_v1.tflite

In [ ]:
# ── 4. Evaluate ───────────────────────────────────────────────────────────────
# Point --test_dir at a held-out split not used during training.
# If using synthetic data, this just confirms the pipeline runs end-to-end.
!python ml/evaluate.py --model neoscreen_v1.tflite --test_dir ./dataset

In [ ]:
# ── 5. Quick single-image smoke test ──────────────────────────────────────────
import numpy as np, tensorflow as tf, cv2

interp = tf.lite.Interpreter('neoscreen_v1.tflite')
interp.allocate_tensors()
inp  = interp.get_input_details()
outp = interp.get_output_details()

# Fake yellow-tinted sclera image
dummy = np.ones((1, 224, 224, 3), dtype=np.float32)
dummy[0,:,:,0] = 1.0   # R
dummy[0,:,:,1] = 0.85  # G
dummy[0,:,:,2] = 0.3   # B  → yellowish

interp.set_tensor(inp[0]['index'], dummy)
interp.invoke()
out = interp.get_tensor(outp[0]['index'])[0]
print(f'P(Low)={out[0]:.3f}  P(Medium)={out[1]:.3f}  P(High)={out[2]:.3f}')
print('Model runs correctly on Colab.')

In [ ]:
# ── 6. Save to Drive & download ───────────────────────────────────────────────
import shutil
shutil.copy('neoscreen_v1.tflite', '/content/drive/MyDrive/neoscreen_v1.tflite')
print('Saved to Drive.')

from google.colab import files
files.download('neoscreen_v1.tflite')
# → Place the downloaded file at: flutter/assets/models/neoscreen_v1.tflite